## DMEPOS - By Referral Provider and Supplier
The purpose of this notebook is to clean the DMEPOS - By Referral Provider and Service and perform EDA on it. Additionally, multiple other datasets will be integrated with this data in the form of flags.

Tasks that were done:
- Integrated and cleaned the 2021–2023 DMEPOS referral provider datasets by standardizing string fields, assessing data quality, handling suppressed values, and adding year and suppression indicators.

- Enriched the data by linking to external sources (LEIE exclusion list and supplier dataset) to flag excluded providers and identify providers who also act as suppliers.

- Performed exploratory analysis and aggregation to compare excluded vs. non-excluded providers across utilization, payment metrics, service categories (RBCS), geography, time, and suppression patterns.

End result:

Saved the cleaned dataset to a shared team directory: /dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr_ser.csv

## Loading the data and quick exploration

In [1]:
# Loading Libraries
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd

In [2]:
# A quick way to see what datasets are available in the team directory
for f in Path("/dsa/groups/casestudycf25/team02/bronze").iterdir():
    print(f.name)

mup_dme_ry25_p05_v10_dy21_rfrhpr.csv
mup_dme_ry25_p05_v10_dy21_rfrr.csv
mup_dme_ry25_p05_v10_dy22_rfrhpr.csv
mup_dme_ry25_p05_v10_dy22_rfrr.csv
mup_dme_ry25_p05_v10_dy23_rfrhpr.csv
mup_dme_ry25_p05_v10_dy23_rfrr.csv
RUCA2010zipcode.csv
DMEPOS_Supplier_Service_2021.csv
DMEPOS_Supplier_Service_2022.csv
DMEPOS_Supplier_Service_2023.csv
leie_with_null_npi_clean.csv
Medicare_Monthly_Enrollment_Jun_2025.csv
CMS_General_Payments_2021_2023_raw.csv
specialty_crosswalk_gpt_backfill.csv
LEIE_OIG_Exclusion_List.csv
mup_dme_ry25_p05_v10_dy23_supr.csv
mup_dme_ry25_p05_v20_dy21_supr.csv
mup_dme_ry25_p05_v20_dy22_supr.csv
mup_dme_ry25_p05_v20_dy21_rfrr.csv
mup_dme_ry25_p05_v20_dy22_rfrr.csv
active_ffs_medicaid_providers.csv
mup_dme_ry25_p05_v10_dy23_suphpr.csv
mup_dme_ry25_p05_v20_dy21_suphpr.csv
mup_dme_ry25_p05_v20_dy22_suphpr.csv


In [3]:
# Loading the raw data
df21 = pd.read_csv(
    "/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy21_rfrhpr.csv",
    dtype={"Suplr_Prvdr_RUCA": "float64"},
    na_values=["NA"]
)

df22 = pd.read_csv(
    "/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy22_rfrhpr.csv",
    dtype={"Suplr_Prvdr_RUCA": "float64"},
    na_values=["NA"]
)

df23 = pd.read_csv(
    "/dsa/groups/casestudycf25/team02/bronze/mup_dme_ry25_p05_v10_dy23_rfrhpr.csv",
    dtype={"Suplr_Prvdr_RUCA": "float64"},
    na_values=["NA"]
)

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)
/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [4]:
#Data Types
df21.dtypes

Rfrg_NPI                      int64
Rfrg_Prvdr_Last_Name_Org     object
Rfrg_Prvdr_First_Name        object
Rfrg_Prvdr_MI                object
Rfrg_Prvdr_Crdntls           object
Rfrg_Prvdr_Ent_Cd            object
Rfrg_Prvdr_St1               object
Rfrg_Prvdr_St2               object
Rfrg_Prvdr_City              object
Rfrg_Prvdr_State_Abrvtn      object
Rfrg_Prvdr_State_FIPS        object
Rfrg_Prvdr_Zip5               int64
Rfrg_Prvdr_RUCA_Cat          object
Rfrg_Prvdr_RUCA             float64
Rfrg_Prvdr_RUCA_Desc         object
Rfrg_Prvdr_Cntry             object
Rfrg_Prvdr_Spclty_Cd         object
Rfrg_Prvdr_Spclty_Desc       object
Rfrg_Prvdr_Spclty_Srce       object
RBCS_Lvl                     object
RBCS_Id                      object
RBCS_Desc                    object
HCPCS_CD                     object
HCPCS_Desc                   object
Suplr_Rentl_Ind              object
Tot_Suplrs                    int64
Tot_Suplr_Benes             float64
Tot_Suplr_Clms              

Similar in structure and attributes to DMEPOS-Supplier & Service dataset

## Carpentry

In [5]:
import re

#2021
# Identify string/object columns
string_cols = [c for c in df21.columns if df21[c].dtype == "object"]

# Transforming all string variables into lowercase, removing special characters and replacing spaces with underscores
for c in string_cols:
    df21[c] = (
        df21[c]
        .astype(str)
        .str.lower()
        .str.replace(r"[^A-Za-z0-9\s]", "", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )

MemoryError: Unable to allocate array with shape (1516153,) and data type float64

In [ ]:
#2022
# Identify string/object columns
string_cols = [c for c in df22.columns if df22[c].dtype == "object"]

# Transforming all string variables into lowercase, removing special characters and replacing spaces with underscores
for c in string_cols:
    df22[c] = (
        df22[c]
        .astype(str)
        .str.lower()
        .str.replace(r"[^A-Za-z0-9\s]", "", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )

In [ ]:
#2023
# Identify string/object columns
string_cols = [c for c in df23.columns if df23[c].dtype == "object"]

# Transforming all string variables into lowercase, removing special characters and replacing spaces with underscores
for c in string_cols:
    df23[c] = (
        df23[c]
        .astype(str)
        .str.lower()
        .str.replace(r"[^A-Za-z0-9\s]", "", regex=True)
        .str.replace(r"\s+", "_", regex=True)
    )

In [ ]:
# Creating a single unified dataset

df23['Year'] = 2023
df22['Year'] = 2022
df21['Year'] = 2021

# Rearranging the columns
order21 = ["Year"] + [c for c in df21.columns if c != "Year"]
order22 = ["Year"] + [c for c in df22.columns if c != "Year"]
order23 = ["Year"] + [c for c in df23.columns if c != "Year"]

df21 = df21[order21]
df22 = df22[order22]
df23 = df23[order23]

df = pd.concat([df21, df22, df23], ignore_index=True)

In [ ]:
# Checking..
df.head()

In [ ]:
# Checking...
df.tail()

In [ ]:
# Shape
df.shape

In [ ]:
# Checking fo null values
nulls = df.isna().sum()
nulls

RUCA-related attributes will be kept for analysis purposes.

In [ ]:
# Suppression indicator for analysis
df["DME_Sprsn_Ind"] = df["Tot_Suplr_Benes"].isna().map({True: "y", False: "n"})
df.head()

In [ ]:
#Add in suppression indicator before total beneficiaries
cols = df.columns.tolist()
cols.remove("DME_Sprsn_Ind")

# Find index of Tot_Suplr_Benes
idx = cols.index("Tot_Suplr_Benes")

# Insert DME_Sprsn_Ind after Tot_Suplr_Benes
cols.insert(idx, "DME_Sprsn_Ind")

# Reorder dataframe
df = df[cols]
df.head()

## EDA

In [ ]:
# Loading the LEIE exclusion list
d = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv",
                dtype={"Suplr_Prvdr_RUCA": "float64"}, na_values=["NA"])

In [ ]:
# Few rows of the cleaned exclusion list
d.head()

In [ ]:
# Flag NPIs in the main dataset that are listed in the LEIE exclusion list
npi_e = d["npi"]
df["Excluded"] = df['Rfrg_NPI'].isin(npi_e).map({True: "y", False: "n"})
df.head()

In [ ]:
# How many NPIs in the main dataset are on the LEIE exclusion list?
df[df['Excluded'] == 'y'].shape[0]

In [ ]:
# Number of unique NPIs in df that are excluded
df[df['Excluded'] == 'y']['Rfrg_NPI'].nunique()

In [ ]:
# Compare the count of unique providers who are excluded versus those who are not
df.groupby('Excluded')['Rfrg_NPI'].nunique()

In [ ]:
# Impact of excluded NPIs on total supplier counts, claims, and beneficiaries
df.groupby('Excluded')[['Tot_Suplrs', 'Tot_Suplr_Clms', 'Tot_Suplr_Benes']].sum()

In [ ]:
# Compare the typical submitted charges and Medicare payment amounts for excluded versus non-excluded NPIs,
df.groupby('Excluded')[['Avg_Suplr_Sbmtd_Chrg', 'Avg_Suplr_Mdcr_Alowd_Amt',
                        'Avg_Suplr_Mdcr_Pymt_Amt', 'Avg_Suplr_Mdcr_Stdzd_Amt']].mean()

In [ ]:
# Which states have the most excluded providers?
df.groupby(['Rfrg_Prvdr_State_Abrvtn', 'Excluded'])['Rfrg_NPI'].nunique().unstack().fillna(0).sort_values(by='y', ascending=False).head(10)

In [ ]:
# Top 10 states with the highest number of referral providers
df.groupby('Rfrg_Prvdr_State_Abrvtn')['Rfrg_NPI'].size().sort_values(ascending=False).head(10)

In [ ]:
# States most affected by LEIE exclusions
df.groupby(['Rfrg_Prvdr_State_Abrvtn', 'Excluded'])['Rfrg_NPI'].size().unstack().fillna(0).sort_values(by='y', ascending=False).head(10)

In [ ]:
# Cities with the largest number of excluded provider records
df[df['Excluded']=='y'].groupby('Rfrg_Prvdr_City').size().sort_values(ascending=False).head(10)

In [ ]:
# Excluded NPIs count over time
ax = (
    df[df['Excluded'] == 'y']
    .groupby(['Year', 'Excluded'])['Rfrg_NPI']
    .nunique()
    .unstack()
    .plot(kind='bar', figsize=(10,5))
)


ax.set_facecolor('white')            
ax.figure.patch.set_facecolor('white')  

ax.set_title("Unique Excluded NPIs by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Unique NPIs")
ax.legend().remove()

In [ ]:
# Count of NPIs excluded each year
df[df['Excluded'] == 'y'].groupby(['Year', 'Excluded'])['Rfrg_NPI'].nunique().unstack()

In [ ]:
#Loading derived supplier dataset
df_sup = pd.read_csv("/dsa/groups/casestudycf25/team02/silver/dmepos_suplr_Bene_only.csv")
df_sup.head()

In [ ]:
# Flagging which referral providers also act as suppliers
npi_s = df_sup['Suplr_NPI']
df["Supplier"] = df['Rfrg_NPI'].isin(npi_s).map({True: "y", False: "n"})
df.head()

In [ ]:
# Distribution of providers acting as suppliers
df['Supplier'].value_counts()

In [ ]:
# Summarizes the relationship between being a supplier and being excluded
pd.crosstab(df['Supplier'], df['Excluded'], margins=True)

In [ ]:
# Summarizes the distribution of suppliers across different RBCS levels
pd.crosstab(df['RBCS_Lvl'], df['Supplier'])

In [ ]:
# Are excluded NPIs also suppressed
suppressed_excluded_count = df[(df["DME_Sprsn_Ind"] == 'y') & (df["Excluded"] == 'y')].shape[0]
suppressed_excluded_count

In [ ]:
# Top 5 states with most suppressed entries
suppressed_by_state = (
    df[df["DME_Sprsn_Ind"] == 'y']
    .groupby("Rfrg_Prvdr_State_Abrvtn")
    .size()
    .reset_index(name="Suppressed_Count")
    .sort_values("Suppressed_Count", ascending=False)
)

suppressed_by_state.head()

In [ ]:
# Suppresed and Excluded by State
excluded_by_state = (
    df[(df["DME_Sprsn_Ind"] == 'y') & (df["Excluded"] == 'y')]
    .groupby("Rfrg_Prvdr_State_Abrvtn")
    .size()
    .reset_index(name="Excluded_Count")
    .sort_values("Excluded_Count", ascending=False)
)
excluded_by_state

In [ ]:
# Top HCPCS Codes among Excluded Providers
plt.figure(figsize=(7,6))
top_hcpcs = df[df['Excluded']=='y'].groupby('HCPCS_CD')['Rfrg_NPI'].count().sort_values(ascending=False).head(10)
top_hcpcs.plot(kind='bar')
plt.title('Top HCPCS Codes among Excluded Providers')
plt.ylabel('Number of Providers')
plt.show()

In [ ]:
# The distribution of excluded vs non-excluded providers across RBCS levels
df.groupby(['RBCS_Lvl', 'Excluded'])['Rfrg_NPI'].count().unstack(fill_value=0)

In [ ]:
#Specialty with most exclusions
df[df['Excluded']=='y']['Rfrg_Prvdr_Crdntls'].value_counts().head(10)

In [ ]:
# Midwest Exclusion

midwest_states = ['mo', 'il', 'ks', 'ia', 'ne', 'mn', 'wi', 'sd', 'nd', 'mi', 'oh', 'in']
df_midwest = df[df['Rfrg_Prvdr_State_Abrvtn'].isin(midwest_states)]

midwest_ratio = df_midwest['Excluded'].value_counts()
print(midwest_ratio)

In [ ]:
#Saving the file
df.to_csv("/dsa/groups/casestudycf25/team02/silver/dmepos_rfrhpr_ser.csv",index=False)